In [ ]:
import boto3
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pickle
import json
from datetime import datetime

# Initialize boto3 clients
s3 = boto3.client('s3')
dynamodb = boto3.resource('dynamodb')

# Load processed data
def load_data_from_s3():
    """Load processed data from S3"""
    print("Loading user_content_matrix...")
    user_content_df = pd.read_parquet('s3://streamify-bucket-lks2026/processed-data/user_content_matrix/')
    print("Loading content_stats...")
    content_stats_df = pd.read_parquet('s3://streamify-bucket-lks2026/processed-data/content_stats/')
    print("Loading user_features...")
    user_features_df = pd.read_parquet('s3://streamify-bucket-lks2026/processed-data/user_features/')
    return user_content_df, content_stats_df, user_features_df

# Content-based filtering
class ContentBasedRecommender:
    def __init__(self):
        self.content_features = None
        self.tfidf_matrix = None
        self.similarity_matrix = None

    def fit(self, content_df):
        """Train content-based model"""
        # Create content features from streaming metadata
        content_df['content'] = (
            content_df['content_type'] + ' ' +
            content_df['genre'] + ' ' +
            content_df['title']
        )
        # TF-IDF Vectorization
        tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
        self.tfidf_matrix = tfidf.fit_transform(content_df['content'])
        # Calculate similarity matrix
        self.similarity_matrix = cosine_similarity(self.tfidf_matrix)
        self.content_features = content_df

    def recommend(self, content_id, n_recommendations=10):
        """Get similar content recommendations"""
        try:
            idx = self.content_features[
                self.content_features['content_id'] == content_id
            ].index[0]
            sim_scores = list(enumerate(self.similarity_matrix[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            similar_content = []
            for i, score in sim_scores[1:n_recommendations+1]:
                similar_content.append({
                    'content_id': self.content_features.iloc[i]['content_id'],
                    'similarity_score': score
                })
            return similar_content
        except:
            return []

# Collaborative filtering
class CollaborativeRecommender:
    def __init__(self):
        self.user_content_matrix = None
        self.svd_model = None
        self.user_embeddings = None
        self.item_embeddings = None

    def fit(self, user_content_df):
        """Train collaborative filtering model"""
        # Create user-content matrix based on stream count
        self.user_content_matrix = user_content_df.pivot_table(
            index='user_id',
            columns='content_id',
            values='stream_count',
            fill_value=0
        )
        # Apply SVD
        self.svd_model = TruncatedSVD(n_components=50, random_state=42)
        self.user_embeddings = self.svd_model.fit_transform(self.user_content_matrix)
        self.item_embeddings = self.svd_model.components_.T

    def recommend(self, user_id, n_recommendations=10):
        """Get recommendations for a user"""
        try:
            user_idx = self.user_content_matrix.index.get_loc(user_id)
            user_vector = self.user_embeddings[user_idx]
            scores = np.dot(user_vector, self.item_embeddings.T)
            top_items = np.argsort(scores)[::-1][:n_recommendations]
            recommendations = []
            for item_idx in top_items:
                content_id = self.user_content_matrix.columns[item_idx]
                score = scores[item_idx]
                recommendations.append({
                    'content_id': content_id,
                    'prediction_score': score
                })
            return recommendations
        except:
            return []

# Hybrid recommender
class HybridRecommender:
    def __init__(self):
        self.content_model = ContentBasedRecommender()
        self.collaborative_model = CollaborativeRecommender()

    def fit(self, user_content_df, content_df):
        """Train both models"""
        self.content_model.fit(content_df)
        self.collaborative_model.fit(user_content_df)

    def recommend(self, user_id, n_recommendations=10):
        """Get hybrid recommendations"""
        collab_recs = self.collaborative_model.recommend(user_id, n_recommendations)
        if not collab_recs:
            return []
        final_recs = []
        for rec in collab_recs[:n_recommendations]:
            final_recs.append({
                'content_id': rec['content_id'],
                'score': rec['prediction_score'] * 0.7,
                'type': 'collaborative'
            })
        return final_recs

# Main training function
def train_recommendation_model():
    """Main training pipeline"""
    print("Loading data...")
    user_content_df, content_stats_df, user_features_df = load_data_from_s3()

    # Load raw content catalog
    print("Loading content catalog...")
    content_df = pd.read_csv('s3://streamify-bucket-lks2026/raw-data/content-catalog/content_catalog.csv')

    print("Training hybrid model...")
    hybrid_model = HybridRecommender()
    hybrid_model.fit(user_content_df, content_df)

    print("Saving model...")
    with open('/tmp/hybrid_model.pkl', 'wb') as f:
        pickle.dump(hybrid_model, f)
    s3.upload_file('/tmp/hybrid_model.pkl', 'streamify-bucket-lks2026', 'models/hybrid_model.pkl')

    print("Generating content embeddings...")
    content_embeddings_table = dynamodb.Table('ContentEmbeddings')

    # ==================== PERBAIKAN UTAMA ====================
    # Cek kolom yang tersedia di content_stats_df
    print("Columns in content_stats_df:", content_stats_df.columns.tolist())
    
    # Tentukan nama kolom untuk total streams
    streams_col = None
    possible_names = ['total_streams', 'stream_count', 'total_streams_count', 'plays', 'total_plays', 'streams']
    for col in possible_names:
        if col in content_stats_df.columns:
            streams_col = col
            break
    
    if streams_col is None:
        print("WARNING: No stream count column found. Using default 0 for popularity.")
        # Tambahkan kolom dummy agar kode tetap berjalan
        content_stats_df['total_streams'] = 0
        streams_col = 'total_streams'
    else:
        print(f"Using column '{streams_col}' for popularity.")
    
    # Loop untuk membuat embedding
    for idx, row in content_df.iterrows():
        content_id = row['content_id']
        
        # Ambil total streams dengan aman
        matching = content_stats_df[content_stats_df['content_id'] == content_id]
        if not matching.empty:
            popularity = float(matching[streams_col].iloc[0])
        else:
            popularity = 0.0
        
        embedding = {
            'content_type': row['content_type'],
            'genre': row['genre'],
            'is_exclusive': bool(row['is_exclusive']),
            'popularity': popularity
        }
        
        content_embeddings_table.put_item(
            Item={
                'content_id': content_id,
                'embedding': json.dumps(embedding),
                'last_updated': datetime.now().isoformat()
            }
        )
        if idx % 100 == 0:
            print(f"Processed {idx} items...")
    
    print("Model training completed!")

# Execute training
if __name__ == "__main__":
    train_recommendation_model()